<a href="https://colab.research.google.com/github/guebin/DL2026/blob/master/posts/11wk.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" style="text-align: left"></a>

# 1. 강의영상

In [1]:
# {{<video https://youtu.be/playlist?list=PLQqh36zP38-xB7p8AzMKUrEfxL0Vte-pY&si=BwVUNxT8IIPrjIcA>}}

# 2. Imports

In [1]:
import torch
import torchvision
import matplotlib.pyplot as plt

In [2]:
plt.rcParams['figure.figsize'] = (4.5, 3.0)

# 3. 다항분류

## A. 범주형자료의 변환 

`-` 범주형자료를 숫자로 어떻게 바꿀까? 

- 실패 / 성공 $\to$ 0 / 1
- 숫자0그림 / 숫자1그림 $\to$ 0 / 1 
- 강아지그림 / 고양이그림 $\to$ 0 / 1
- 강아지그림 / 고양이그림 / 토끼그림 $\to$ 0 / 1 / 2 ?????

`-` 주입식교육: 강아지그림/고양이그림/토끼그림일 경우 숫자화시키는 방법 

- 잘못된방식: 강아지그림 = 0, 고양이그림 = 1, 토끼그림 = 2
- 올바른방식: 강아지그림 = [1,0,0], 고양이그림 = [0,1,0], 토끼그림 = [0,0,1]  ### <-- 이런방식을 원핫인코딩이라함

`-` 왜? 

- 설명1: 강아지그림, 고양이그림,  토끼그림은 서열측도가 아니라 명목척도임. 그래서 범주를 0,1,2 로 숫자화하면 평균등의 의미가 없음 (사회조사분석사 2급 스타일) 
- 설명2: 범주형은 원핫인코딩으로 해야함 ("30일만에 끝내는 실전머신러닝" 이런 책에 나오는 스타일) 
- 설명3: 동전을 한번 던져서 나오는 결과는 베르누이분포, 즉 시행횟수=1인 이항분포 $B(1,p)$를 따름. 주사위 한번 던져서 나오는 눈금의 숫자는 시행횟수=1인 다항분포 $M(1,{\bf p})$를 따름. 시행횟수=1인 이항분포의 실현값은 0,1 이고, 시행횟수=1인 다항분포의 실현값은 [1,0,0], [0,1,0], [0,0,1] 이므로 당연히 $y_i$ 는 [1,0,0], [0,1,0], [0,0,1] 중 하나의 형태를 가진다고 가정하는게 바람직함 (이 설명이 이 중에서 가장 정확한 설명임)

## B. 실습: 3개의 클래스를 구분

`-` 데이터준비 

In [31]:
train_dataset = torchvision.datasets.MNIST(root='./data',train=True,download=True)
to_tensor = torchvision.transforms.ToTensor()
X3 = torch.stack([to_tensor(Xi) for Xi, yi in train_dataset if yi==3])
X7 = torch.stack([to_tensor(Xi) for Xi, yi in train_dataset if yi==7])
X9 = torch.stack([to_tensor(Xi) for Xi, yi in train_dataset if yi==9])
X = torch.concat([X3,X7,X9]).reshape(-1,1*28*28)
y = torch.tensor([0]*6131 + [1]*6265+ [2]*len(X9))

In [33]:
print(X.shape)
print(y.shape)

torch.Size([18345, 784])
torch.Size([18345])


In [35]:
y # 왜 이렇게??

tensor([0, 0, 0,  ..., 2, 2, 2])

In [37]:
Y = torch.nn.functional.one_hot(y).float()
Y

tensor([[1., 0., 0.],
        [1., 0., 0.],
        [1., 0., 0.],
        ...,
        [0., 0., 1.],
        [0., 0., 1.],
        [0., 0., 1.]])

`-` 적합 

In [40]:
net = torch.nn.Sequential(
    torch.nn.Linear(784,32),
    torch.nn.ReLU(),
    torch.nn.Linear(32,3),
)
loss_fn = torch.nn.CrossEntropyLoss() # 의미상 CEWithLogitsLoss
optimizr = torch.optim.Adam(net.parameters())
for epoc in range(1,31):
    #1
    netout = net(X) # netout: (n,3) 
    #2
    loss = loss_fn(netout,Y) 
    #3
    loss.backward()
    #4 
    optimizr.step()
    optimizr.zero_grad()

In [42]:
(netout.argmax(axis=1) == Y.argmax(axis=1)).float().mean()

tensor(0.9277)

## C. 결론 -- 외우세여

`-` 파이토치버전 // 코딩용

|task | netout의 의미 | 손실함수 | 
|:--:|:--:|:--:|
|이항분류 | prob | `BCELoss` | 
|이항분류 | logit | `BCEWithLogitsLoss` | 
|다항분류| probs |  NA |
|다항분류| logits |  `CrossEntropyLoss` | 

> `CrossEntropyLoss` 이거 이름이 완전 마음에 안들어요.. `CEWithLogitsLoss` 라고 하는게 더 좋을 것 같습니다. 

`-` 일반적개념 // 이론용

|task |오차항의가정|  마지막활성화함수 | 손실함수 | 
|:--:|:--:|:--:|:--:|
|이항분류 | 시행횟수=1인 이항분포 | Sigmoid | BCE | 
|다항분류| 시행횟수=1인 다항분포 | Softmax | CE |

`-` 참고 (sigmoid, softmax 계산과정비교) 

- $prob = \text{sig}(logit) =\frac{\exp(logit)}{1+\exp(logit)}$

- ${\bf probs}= \text{softmax}({\bf logtis}) = \text{softmax}\left(\begin{bmatrix} score_1 \\ score_2 \\ score_3\end{bmatrix}\right) 
=\begin{bmatrix} 
\frac{\exp(score_1)}{\exp(score_1)+\exp(score_2)+\exp(score_3)} \\ 
\frac{\exp(score_2)}{\exp(score_1)+\exp(score_2)+\exp(score_3)} \\
\frac{\exp(score_3)}{\exp(score_1)+\exp(score_2)+\exp(score_3)}
\end{bmatrix}$

# 4. CNN 자랑

## A. 성능좋음

*Fashion MNIST*

In [21]:
train_dataset = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True)
test_dataset = torchvision.datasets.FashionMNIST(root='./data', train=False, download=True)
train_dataset = torch.utils.data.Subset(train_dataset, range(5000))
test_dataset = torch.utils.data.Subset(test_dataset, range(1000))
to_tensor = torchvision.transforms.ToTensor()
X = torch.stack([to_tensor(img) for img, lbl in train_dataset]).to("cuda:0")
y = torch.tensor([lbl for img, lbl in train_dataset])
y = torch.nn.functional.one_hot(y).float().to("cuda:0")
XX = torch.stack([to_tensor(img) for img, lbl in test_dataset]).to("cuda:0")
yy = torch.tensor([lbl for img, lbl in test_dataset])
yy = torch.nn.functional.one_hot(yy).float().to("cuda:0")

*발악수준으로 설계한 신경망*

In [31]:
torch.manual_seed(0)
net = torch.nn.Sequential(
    torch.nn.Flatten(),
    torch.nn.Linear(784,2048),
    torch.nn.ReLU(),
    torch.nn.Linear(2048,10)
).to("cuda")
loss_fn = torch.nn.CrossEntropyLoss()
optimizr = torch.optim.Adam(net.parameters())

In [32]:
for epoc in range(1,500):
    #1
    logits = net(X)
    #2
    loss = loss_fn(logits, y) 
    #3
    loss.backward()
    #4 
    optimizr.step()
    optimizr.zero_grad()

In [33]:
(net(X).argmax(axis=1) == y.argmax(axis=1)).float().mean()

tensor(1., device='cuda:0')

In [34]:
(net(XX).argmax(axis=1) == yy.argmax(axis=1)).float().mean()

tensor(0.8530, device='cuda:0')

*대충대충 설계한 합성곱신경망*

In [35]:
torch.manual_seed(0)
net = torch.nn.Sequential(
    torch.nn.Conv2d(1,16,2),
    torch.nn.ReLU(),
    torch.nn.MaxPool2d(2),
    torch.nn.Flatten(),
    torch.nn.Linear(2704,10),
).to("cuda")
loss_fn = torch.nn.CrossEntropyLoss()
optimizr = torch.optim.Adam(net.parameters())

In [36]:
for epoc in range(1,500):
    #1
    logits = net(X)
    #2
    loss = loss_fn(logits, y) 
    #3
    loss.backward()
    #4 
    optimizr.step()
    optimizr.zero_grad()

In [37]:
(net(X).argmax(axis=1) == y.argmax(axis=1)).float().mean()

tensor(0.9670, device='cuda:0')

In [38]:
(net(XX).argmax(axis=1) == yy.argmax(axis=1)).float().mean()

tensor(0.8730, device='cuda:0')

## B. 파라메터적음

In [39]:
net1 = torch.nn.Sequential(
    torch.nn.Flatten(),
    torch.nn.Linear(784,2048),
    torch.nn.ReLU(),
    torch.nn.Linear(2048,10)
)
net2 = torch.nn.Sequential(
    torch.nn.Conv2d(1,16,2),
    torch.nn.ReLU(),
    torch.nn.MaxPool2d(2),
    torch.nn.Flatten(),
    torch.nn.Linear(2704,10),
)

In [40]:
net1_params = list(net1.parameters())
print(net1_params[0].shape)
print(net1_params[1].shape)
print(net1_params[2].shape)
print(net1_params[3].shape)

torch.Size([2048, 784])
torch.Size([2048])
torch.Size([10, 2048])
torch.Size([10])


In [41]:
2048*784 + 2048 + 10*2048 + 10 

1628170

In [42]:
net2_params = list(net2.parameters())
print(net2_params[0].shape)
print(net2_params[1].shape)
print(net2_params[2].shape)
print(net2_params[3].shape)

torch.Size([16, 1, 2, 2])
torch.Size([16])
torch.Size([10, 2704])
torch.Size([10])


In [43]:
16*1*2*2 + 16 + 10*2704 + 10 

27130

In [44]:
27130/1628170

0.01666287918337766

## C. 유명함

`-` <https://brunch.co.kr/@hvnpoet/109> 

# 5. CNN 핵심레이어 

## A. `torch.nn.ReLU`

**(예시1) 연산방법**

In [39]:
img = torch.randn(1,1,4,4) # (4,4) 흑백이미지 한장
relu = torch.nn.ReLU()

In [42]:
img

tensor([[[[ 0.0052,  1.1922,  0.7636,  0.0099],
          [-0.9365,  0.0695, -0.1974, -0.1691],
          [-1.9972,  0.9638, -0.8581, -1.1956],
          [ 1.2276,  0.9221,  1.3697, -0.2663]]]])

In [43]:
relu(img)

tensor([[[[0.0052, 1.1922, 0.7636, 0.0099],
          [0.0000, 0.0695, 0.0000, 0.0000],
          [0.0000, 0.9638, 0.0000, 0.0000],
          [1.2276, 0.9221, 1.3697, 0.0000]]]])

## B. `torch.nn.MaxPool2d`

**(예시1) 연산방법, kernel_size 의 의미**

In [57]:
img = torch.rand(1,1,4,4)
mp = torch.nn.MaxPool2d(kernel_size=2)

In [58]:
img

tensor([[[[0.4601, 0.1505, 0.8785, 0.2573],
          [0.4426, 0.5923, 0.4630, 0.9225],
          [0.9051, 0.5439, 0.8494, 0.6388],
          [0.9822, 0.1382, 0.6126, 0.9961]]]])

In [59]:
mp(img)

tensor([[[[0.5923, 0.9225],
          [0.9822, 0.9961]]]])

**(예시2) 이미지크기와 딱 맞지않는 커널일경우?**

In [63]:
img = torch.rand(1,1,5,5)
mp = torch.nn.MaxPool2d(kernel_size=3)

In [64]:
img

tensor([[[[0.4661, 0.5162, 0.0087, 0.7542, 0.1391],
          [0.0969, 0.5140, 0.3865, 0.1853, 0.5127],
          [0.7183, 0.3710, 0.5541, 0.1578, 0.4765],
          [0.6287, 0.1574, 0.6492, 0.9207, 0.5921],
          [0.7354, 0.9558, 0.8880, 0.9573, 0.7333]]]])

In [65]:
mp(img)

tensor([[[[0.7183]]]])

**(예시3) 정사각형이 아닌 커널**

In [69]:
img = torch.rand(1,1,4,4)
mp = torch.nn.MaxPool2d(kernel_size=(4,2))

In [70]:
img

tensor([[[[0.9469, 0.4182, 0.0710, 0.1394],
          [0.2413, 0.7493, 0.0440, 0.7918],
          [0.9179, 0.8230, 0.0547, 0.4162],
          [0.9223, 0.5011, 0.8517, 0.9853]]]])

In [72]:
mp(img)

tensor([[[[0.9469, 0.9853]]]])

## C. `torch.nn.Conv2d`

**(예시1) 연산방법, stride=2**

In [4]:
img = torch.rand(1,1,4,4)
conv = torch.nn.Conv2d(in_channels=1,out_channels=1,kernel_size=2,stride=2)

In [5]:
img

tensor([[[[0.3041, 0.9222, 0.4187, 0.7885],
          [0.7488, 0.5487, 0.7938, 0.0053],
          [0.5002, 0.8596, 0.9975, 0.7992],
          [0.7594, 0.3295, 0.9948, 0.2643]]]])

In [6]:
conv(img)

tensor([[[[0.4708, 0.6137],
          [0.5387, 0.6470]]]], grad_fn=<ConvolutionBackward0>)

??

In [7]:
conv.weight.data, conv.bias.data

(tensor([[[[-0.2970, -0.3377],
           [ 0.0756, -0.1169]]]]),
 tensor([0.4719]))

In [8]:
(img[:,  :,  :2,  :2] * conv.weight.data).sum()+conv.bias.data, conv(img)

(tensor([0.1281]),
 tensor([[[[0.1281, 0.0595],
           [0.1877, 0.0775]]]], grad_fn=<ConvolutionBackward0>))

In [9]:
(img[:,  :,  :2,  2:] * conv.weight.data).sum()+conv.bias.data, conv(img)

(tensor([0.0595]),
 tensor([[[[0.1281, 0.0595],
           [0.1877, 0.0775]]]], grad_fn=<ConvolutionBackward0>))

In [10]:
(img[:,  :,  :2,  2:] * conv.weight.data).sum()+conv.bias.data, conv(img)

(tensor([0.0595]),
 tensor([[[[0.1281, 0.0595],
           [0.1877, 0.0775]]]], grad_fn=<ConvolutionBackward0>))

In [11]:
(img[:,  :,  2:,  2:] * conv.weight.data).sum()+conv.bias.data, conv(img)

(tensor([0.0775]),
 tensor([[[[0.1281, 0.0595],
           [0.1877, 0.0775]]]], grad_fn=<ConvolutionBackward0>))

:::{.callout-note}

**입력이 1장의 흑백이미지이고 출력도 1장의 흑백이미지일 경우 컨볼루션 계산과정 요약^[입력shape=(1,1,?,?) 이고 출력의shape=(1,1,?,?)일 경우]**



1. 윈도우생성: `kernel_size` = (?,?) 인 윈도우를 만듦
2. sub-img생성: 입력 이미지에 윈도우를 통과시켜 (?,?) 크기의 sub-img를 만듦.
3. 연산: sub-img의 각 원소에 `conv.weight`의 값을 원소별로 (=element-wisely) 곱하고 결과를 더함. (만약에 `conv.bias`가 있다면 최종결과에 bias를 더함) 
4. 이동&반복: 윈도우를 `stride` 만큼 이동하여 반복. (`stride`=1 이라면 한칸씩, `stride`=2 라면 두칸씩 이동) 

:::

**(예시2)** -- 재현 

"A guide to convolution arithmetic for deep learning"  [@dumoulin2016guide] 에 나온 그림재현 

- ref: <https://arxiv.org/abs/1603.07285> 

![Fig: conv2d 계산과정시각화](https://github.com/guebin/2021BDA/blob/master/_notebooks/2021-11-04-conv.png?raw=true)

In [18]:
img = torch.tensor([
    [3,3,2,1,0],
    [0,0,1,3,1],
    [3,1,2,2,3],
    [2,0,0,2,2],
    [2,0,0,0,1]
]).reshape(1,1,5,5).float()
img

tensor([[[[3., 3., 2., 1., 0.],
          [0., 0., 1., 3., 1.],
          [3., 1., 2., 2., 3.],
          [2., 0., 0., 2., 2.],
          [2., 0., 0., 0., 1.]]]])

In [17]:
conv = torch.nn.Conv2d(in_channels=1,out_channels=1,kernel_size=3,bias=False)
conv.weight.data = torch.tensor([[[
    [ 0.0, 1.0, 2.0],
    [ 2.0, 2.0, 0.0],
    [ 0.0, 1.0, 2.0]
]]])

In [19]:
conv(img)

tensor([[[[12., 12., 17.],
          [10., 17., 19.],
          [ 9.,  6., 14.]]]], grad_fn=<ConvolutionBackward0>)

**(예시3) 이동평균**

In [33]:
img = torch.arange(1,17).float().reshape(1,1,4,4)
img

tensor([[[[ 1.,  2.,  3.,  4.],
          [ 5.,  6.,  7.,  8.],
          [ 9., 10., 11., 12.],
          [13., 14., 15., 16.]]]])

In [27]:
conv = torch.nn.Conv2d(in_channels=1,out_channels=1,kernel_size=2,stride=1,bias=False)
conv.weight.data = conv.weight.data*0 + 1/4
conv.weight.data

tensor([[[[0.2500, 0.2500],
          [0.2500, 0.2500]]]])

In [34]:
conv(img)

tensor([[[[ 3.5000,  4.5000,  5.5000],
          [ 7.5000,  8.5000,  9.5000],
          [11.5000, 12.5000, 13.5000]]]], grad_fn=<ConvolutionBackward0>)

**(예시4) 2개의 이미지**

`-` 개념: (1,1,?,?) $\to$ (1,1,?,?) 의 conv를 observation 별로 적용 

- `conv` 에 포함된 파라메터 수는 (1,1,?,?) $\to$ (1,1,?,?) 인 경우와 (n,1,?,?) $\to$ (n,1,?,?)인 경우가 동일 

In [996]:
imgs = torch.arange(1,33).float().reshape(2,1,4,4)
conv = torch.nn.Conv2d(in_channels=1,out_channels=1,kernel_size=2,stride=1,bias=False)
conv.weight.data = conv.weight.data*0 + 1/4

In [37]:
imgs

tensor([[[[ 1.,  2.,  3.,  4.],
          [ 5.,  6.,  7.,  8.],
          [ 9., 10., 11., 12.],
          [13., 14., 15., 16.]]],


        [[[17., 18., 19., 20.],
          [21., 22., 23., 24.],
          [25., 26., 27., 28.],
          [29., 30., 31., 32.]]]])

In [39]:
conv(imgs)

tensor([[[[ 3.5000,  4.5000,  5.5000],
          [ 7.5000,  8.5000,  9.5000],
          [11.5000, 12.5000, 13.5000]]],


        [[[19.5000, 20.5000, 21.5000],
          [23.5000, 24.5000, 25.5000],
          [27.5000, 28.5000, 29.5000]]]], grad_fn=<ConvolutionBackward0>)

**(예시5) 2개의 이미지, 2개의 out_channels**

`-` 개념: (1,1,?,?) $\to$ (1,1,?,?) 의 conv를 한번 적용, 그것과 별개로 (1,1,?,?) $\to$ (1,1,?,?) 인 **다른** conv를 적용함. (즉 하나의 observation당 2번 conv변환) 이것을 observation별로 반복

- (1,1,?,?) $\to$ (1,2,?,?) 인 경우는 (1,1,?,?) $\to$ (1,1,?,?)인 경우보다 `conv`에 포함된 파라메터 수가 2배 많음
- 그런데 (1,1,?,?) $\to$ (1,2,?,?) 인 경우와 (n,1,?,?) $\to$ (n,2,?,?)인 경우는 `conv`에 포함된 파라메터 수가 같음.
- 따라서 (n,1,?,?) $\to$ (n,2,?,?) 인 경우는 (1,1,?,?) $\to$ (1,1,?,?)인 경우보다 `conv`에 포함된 파라메터 수가 2배 많음

In [92]:
img = torch.arange(1,33).float().reshape(2,1,4,4)
conv = torch.nn.Conv2d(in_channels=1,out_channels=2,kernel_size=2,stride=1,bias=False)

In [93]:
img

tensor([[[[ 1.,  2.,  3.,  4.],
          [ 5.,  6.,  7.,  8.],
          [ 9., 10., 11., 12.],
          [13., 14., 15., 16.]]],


        [[[17., 18., 19., 20.],
          [21., 22., 23., 24.],
          [25., 26., 27., 28.],
          [29., 30., 31., 32.]]]])

In [94]:
conv.weight.data[0] = conv.weight.data[0]*0 +1/4
conv.weight.data[1] = conv.weight.data[0]*0

In [95]:
conv(img)

tensor([[[[ 3.5000,  4.5000,  5.5000],
          [ 7.5000,  8.5000,  9.5000],
          [11.5000, 12.5000, 13.5000]],

         [[ 0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000,  0.0000]]],


        [[[19.5000, 20.5000, 21.5000],
          [23.5000, 24.5000, 25.5000],
          [27.5000, 28.5000, 29.5000]],

         [[ 0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000,  0.0000]]]], grad_fn=<ConvolutionBackward0>)

In [80]:
conv(img)

tensor([[[[ 1.2309,  1.2712,  1.3115],
          [ 1.3920,  1.4322,  1.4725],
          [ 1.5530,  1.5933,  1.6335]],

         [[ 3.9083,  4.5405,  5.1728],
          [ 6.4372,  7.0695,  7.7017],
          [ 8.9662,  9.5984, 10.2307]]],


        [[[ 1.8751,  1.9154,  1.9556],
          [ 2.0362,  2.0764,  2.1167],
          [ 2.1972,  2.2375,  2.2777]],

         [[14.0241, 14.6563, 15.2886],
          [16.5530, 17.1853, 17.8175],
          [19.0820, 19.7142, 20.3465]]]], grad_fn=<ConvolutionBackward0>)